# Mini-Project 2: Uganda Tax Engine

This notebook demonstrates the reusable OOP model in `src/tax_engine.py`. All amounts are exact `Decimal`-backed UGX. The PAYE resident bands below implement the assignment PDF's stated baseline; this is coursework, not tax advice.

The URA PAYE pages redirected to a generic customer portal when checked on 14 October 2026. The assignment baseline is therefore implemented as provided, with the non-resident no-tax-free-threshold rule noted from the URA page listing. Verify current legislation and rates before real payroll use. The modeled schedule date is 1 July 2026 solely for this assignment.

In [1]:
from datetime import date
from decimal import Decimal
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.tax_engine import (
    CorporateIncomeTax, Invoice, InvalidIncomeError, LocalServiceTax, Money,
    NoScheduleForDateError, PAYE, Payroll, SupplyType, Tax, TaxContext,
    TaxPlanner, TaxScenario, VAT,
)

print(f"Schedule modeled from {date(2026, 7, 1)}; amounts are UGX rounded half-up to whole shillings.")

Schedule modeled from 2026-07-01; amounts are UGX rounded half-up to whole shillings.


## Money and the abstract tax contract

`Tax` defines one calculation interface. PAYE, NSSF, Local Service Tax, VAT, and corporate tax implement it; payroll composes these calculators rather than inheriting from them. `Money` accepts decimal strings, integers, or `Decimal`, rejects floats, and rounds explicitly using `ROUND_HALF_UP`.

In [2]:
float_sum = 0.1 + 0.2
decimal_sum = Decimal("0.1") + Decimal("0.2")
print(f"Float: {float_sum!r}; exact Decimal: {decimal_sum}")
assert float_sum != 0.3
assert decimal_sum == Decimal("0.3")

salary = Money.of("1500000")
allowance = Money.of("200000")
print(f"Money addition: {salary + allowance}; 5%: {salary * Decimal('0.05')}")
print(f"Comparison: {Money.of(100)} < {Money.of(200)} is {Money.of(100) < Money.of(200)}")

Float: 0.30000000000000004; exact Decimal: 0.3
Money addition: UGX 1,700,000; 5%: UGX 75,000
Comparison: UGX 100 < UGX 200 is True


## PAYE bands and boundary checks

The resident bands and marginal rates match the figures in the assignment: nil through UGX 335,000; 20% to 410,000; 25% to 485,000; 30% to 10,000,000; then the additional 10% surcharge on excess above 10,000,000. The exact edges below are also tested in `tests/test_tax_engine.py`.

In [3]:
paye = PAYE()
boundary_inputs = [335_000, 335_001, 410_000, 485_000, 10_000_000, 10_000_001]
boundary_expected = [0, 0, 15_000, 33_750, 2_888_250, 2_888_250]
boundary_rows = []
for income, expected in zip(boundary_inputs, boundary_expected):
    tax = paye.calculate(TaxContext(amount=Money.of(income))).amount
    assert tax == Money.of(expected)
    boundary_rows.append({"Monthly income (UGX)": income, "PAYE (UGX)": tax.amount})

boundary_table = pd.DataFrame(boundary_rows)
print(boundary_table.to_string(index=False))
nonresident_tax = paye.calculate(TaxContext(amount=Money.of(335_000), resident=False)).amount
assert nonresident_tax == Money.of(33_500)
print(f"Non-resident PAYE at UGX 335,000: {nonresident_tax}")

 Monthly income (UGX) PAYE (UGX)
               335000          0
               335001          0
               410000      15000
               485000      33750
             10000000    2888250
             10000001    2888250
Non-resident PAYE at UGX 335,000: UGX 33,500


The non-resident schedule applies 10% from the first shilling, so there is no tax-free threshold. For this teaching implementation, the later marginal thresholds and rates use the assignment bands. The URA page listing confirms the no-threshold treatment, but the current full schedule could not be retrieved during this work; this schedule assumption must not be treated as verified law. The `effective_date` check raises `NoScheduleForDateError` for dates before the single schedule modeled here.

In [4]:
try:
    paye.calculate(TaxContext(amount=Money.of(500_000), effective_date=date(2025, 7, 1)))
except NoScheduleForDateError as error:
    print(f"Historical date rejected explicitly: {error}")

try:
    paye.calculate(TaxContext(amount=Money.of(100), deductions=Money.of(200)))
except InvalidIncomeError as error:
    print(f"Invalid taxable income rejected explicitly: {error}")

Historical date rejected explicitly: No schedule configured for 2025-07-01
Invalid taxable income rejected explicitly: Allowances and income cannot be less than deductions


## Payroll pipeline

The pipeline calculates employee NSSF at 5% of base gross pay and reports the employer's 10% separately. It subtracts configured Local Service Tax and any explicit deductions from gross pay plus allowances before PAYE, following the ordering required by the assignment. Net pay then subtracts employee NSSF, LST, and PAYE from gross pay plus allowances. Employer NSSF is an employer cost and is not subtracted from employee take-home pay. The UGX 5,000 LST below is an example configuration, not a statutory schedule.

In [5]:
payroll = Payroll(lst=LocalServiceTax(Money.of(5_000)))
payroll_result = payroll.calculate(
    TaxContext(amount=salary, allowances=allowance, resident=True)
)
for label, amount in payroll_result.__dict__.items():
    print(f"{label.replace('_', ' ').title():22} {amount}")
assert payroll_result.net_pay < payroll_result.gross

Gross                  UGX 1,700,000
Nssf Employee          UGX 75,000
Nssf Employer          UGX 150,000
Local Service Tax      UGX 5,000
Paye                   UGX 396,750
Net Pay                UGX 1,223,250


## VAT and corporate income tax

Each invoice carries a supply type and purchase/sale flag. Standard-rated sales create output VAT at 18%; standard-rated purchases create recoverable input VAT. Zero-rated and exempt invoices create no VAT in this simplified model, and exempt purchases do not recover input VAT. Net VAT payable is output less recoverable input VAT. Corporate chargeable income is revenue minus allowable deductions, floored at zero by validation rather than silently calculating a negative tax base.

In [6]:
invoices = (
    Invoice(Money.of(100_000), SupplyType.STANDARD),
    Invoice(Money.of(50_000), SupplyType.ZERO_RATED),
    Invoice(Money.of(75_000), SupplyType.EXEMPT),
    Invoice(Money.of(20_000), SupplyType.STANDARD, is_purchase=True),
)
vat_result = VAT().calculate(TaxContext(invoices=invoices))
print(f"Output VAT: {vat_result.breakdown['output_vat']}")
print(f"Recoverable input VAT: {vat_result.breakdown['input_vat']}")
print(f"Net VAT payable: {vat_result.amount}")
assert vat_result.amount == Money.of(14_400)

corporate_result = CorporateIncomeTax().calculate(
    TaxContext(amount=Money.of(1_000_000), deductions=Money.of(200_000))
)
print(f"Corporate chargeable income: {corporate_result.breakdown['chargeable_income']}")
print(f"Corporate income tax at 30%: {corporate_result.amount}")
assert corporate_result.amount == Money.of(240_000)

Output VAT: UGX 18,000
Recoverable input VAT: UGX 3,600
Net VAT payable: UGX 14,400
Corporate chargeable income: UGX 800,000
Corporate income tax at 30%: UGX 240,000


## Extension: TaxPlanner

The planner compares two already-calculated scenarios and reports the take-home difference; it does not guess which tax law applies. Comparing resident and non-resident treatment for the same gross salary makes the assumed schedule difference visible. A production salary-versus-dividend comparison would need verified dividend withholding, social contributions, and eligibility rules for the relevant taxpayer and year.

In [7]:
comparison = TaxPlanner().compare(
    TaxScenario("Resident PAYE", salary, paye.calculate(TaxContext(amount=salary)).amount),
    TaxScenario("Non-resident PAYE", salary, paye.calculate(TaxContext(amount=salary, resident=False)).amount),
)
print(f"{comparison.first.name} take-home: {comparison.first.take_home}")
print(f"{comparison.second.name} take-home: {comparison.second.take_home}")
print(f"Difference (first minus second): {comparison.difference}")

Resident PAYE take-home: UGX 1,161,750
Non-resident PAYE take-home: UGX 1,128,250
Difference (first minus second): UGX 33,500


## Discussion and sources

**Why not float?** Binary floating-point stores many decimal fractions approximately. In Python, `0.1 + 0.2` is `0.30000000000000004`, not exactly `0.3`. Repeating percentage calculations near a threshold can preserve tiny errors until rounding, potentially changing a shilling at a boundary. `Decimal` constructed from a string represents decimal inputs exactly; this project rounds each UGX result to a whole shilling using `ROUND_HALF_UP`. Creating `Decimal` from a float would preserve the float's approximation, so `Money` rejects floats.

**Where is inheritance justified?** The abstract `Tax` base makes the required `calculate(TaxContext) -> TaxResult` contract explicit and lets distinct tax modules be used consistently. `PAYE`, `NSSF`, `LocalServiceTax`, `VAT`, and `CorporateIncomeTax` implement that contract. `Payroll` uses composition: it owns PAYE, NSSF, and configurable LST calculators and coordinates their order. `Money`, invoices, contexts, and payroll results are values, not tax algorithms, so making them inherit from `Tax` would add no useful substitutability.

The code uses only Python's standard `decimal`, `dataclasses`, `enum`, and `abc` modules plus pandas for the compact boundary table; pytest and Hypothesis are limited to tests. Property-based checks require PAYE to remain non-negative, monotonic, and no greater than income across generated non-negative incomes. Boundary assertions cover all six required values. These properties test the modeled arithmetic, not whether the modeled rates are current law.

**Rate scope and sources.** The resident monthly PAYE amounts are copied from the Advent 2026 assignment baseline: nil up to UGX 335,000; then marginal rates of 20%, 25%, and 30% over the listed bands; above UGX 10,000,000 the extra 10% surcharge applies to the excess. Assignment-provided baselines also supply employee/employer NSSF of 5%/10%, VAT standard rate 18%, and resident-company corporate rate 30%. Local Service Tax is configurable because the assignment describes it as optional. On 14 October 2026, the official URA PAYE and employment-income URLs redirected to a generic customer portal, so the current full schedule and NSSF details could not be independently confirmed from page content. The URA PAYE search listing stated that non-residents are not entitled to a tax-free threshold; this notebook models 10% from the first shilling and applies the course baseline's later marginal bands as an explicit assumption. The schedule begins 1 July 2026 to demonstrate date rejection, not because a URA document verified that effective date.

References checked 14 October 2026: [URA PAYE rates](https://www.ura.go.ug/en/domestic-taxes/paye-rates/), [URA employment income](https://www.ura.go.ug/en/employment-income/), [Uganda Income Tax Act, Chapter 338, ULII version dated 23 December 2024](https://ulii.org/akn/ug/act/1997/11/eng@2024-12-23), and [NSSF Uganda](https://www.nssfug.org/). Check the current Act, amendments, and URA guidance before applying these illustrative rules to payroll or tax filing.